# PCAs

This is a notebook to see if the neighborhood is homogeneous (using PCA and visual inspection). In other words, do the samples close in the embedding space have the same label?

In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import polars as pl
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import colorcet as cc

from project.utils.strs import SEED, plms, pca_figures_dir, pca_compiled_results_dir
from project.utils.functions import (load_config, 
                                    filter_onehot_df, 
                                    get_cached_embeddings, 
                                    pca_layerwise_embeddings,
                                    get_embeddings, 
                                    scramble_sequences)

from project.algorithms.networks.protein_language_model import ProteinLanguageModel

In [ ]:
# Make dirs if they don't exist already
for directory in [pca_figures_dir, pca_compiled_results_dir]:
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
# from pathlib import Path
# import matplotlib.pyplot as plt
# import seaborn as sns
# import pandas as pd
# import numpy as np
# import polars as pl
# import colorcet as cc
# from matplotlib.lines import Line2D
# from project.utils.functions import (load_config, 
#                                     filter_onehot_df, 
#                                     get_cached_embeddings, 
#                                     pca_layerwise_embeddings,
#                                     get_embeddings, 
#                                     scramble_sequences)
# from project.utils.strs import SEED, plms
# 

### PCA

Here, we don't want to have too many labels (too noisy) so we'll pick proteins with 1-3 labels for each dataset. We'll use scrambled sequences as a control, and use cached embeddings where possible (generated elsewhere, in scripts/embeddings).

In [ ]:
annotation_min_overlap = 1
annotation_max_overlap = 3
num_straps = 3
num_scrambled = 50

model_shorthands = ['amplify_120m', 'amplify_350m', 'esm2_150m'] #'esm2_8m', 'esm2_35m',  'esm2_650m',   'amplify_350m'
datasets = ['interpro_conserved_site', 
            'interpro_repeat',
            'interpro_active_site',
            'interpro_domain', 
            'interpro_family', 
            'interpro_homologous_superfamily', 
            'GO_cc',
            'GO_mf', 
            'GO_bp',]
layer_nums = [0,5,10,15,20]

layer_dfs = []

for dataset in datasets:
    # Load dataset
    df = pl.read_parquet(load_config(dataset)['data_path'])

    #Add the annotation column, which is just basically a string join of positive columns (we specify the number of positive labels we want)
    annotation_df, label_dict, le = filter_onehot_df(df, 
                    annotation_min_overlap=annotation_min_overlap, 
                    annotation_max_overlap=annotation_max_overlap,
                    min_label_count=num_straps+1)

    encoded_label_col = annotation_df['label_num'].to_numpy()
    label_col = annotation_df['combined_labels'].to_list()
    sequences = annotation_df['sequence'].to_list()

    # Sample a set of protein sequences to scramble
    to_scramble = annotation_df.sample(num_scrambled, seed=SEED)
    scrambled_sequences = scramble_sequences(to_scramble['sequence'].to_list(), seed=SEED)
    scrambled_seq_labels = ['scrambled' for time in range(num_scrambled)]
    highest_label_num = max(encoded_label_col) + 1
    scrambled_seq_labels_encoded = [highest_label_num for time in range(num_scrambled)]

    scramble_df = pl.DataFrame({
        'sequence': pl.Series(name='sequence', values=scrambled_sequences, dtype=pl.String),
        'combined_labels': pl.Series(name='combined_labels', values=scrambled_seq_labels, dtype=pl.String),
        'label_num': pl.Series(name='label_num', values=scrambled_seq_labels_encoded, dtype=pl.Int64),
        }
    )

    annotation_df = pl.concat([annotation_df, scramble_df], how='diagonal_relaxed')

    for model_shorthand in model_shorthands:
        # Load cached embeddings
        embeddings = get_cached_embeddings(sequences = sequences, model_shorthand=model_shorthand, layer_nums = layer_nums, as_numpy=True)

        # Get embeddings for scrambled sequences
        plm = ProteinLanguageModel(plms[model_shorthand]['full_name'], layer_to_use=None)
        scrambled_embeddings = get_embeddings(sequences = scrambled_sequences, 
                                            batch_size=4,
                                            protein_language_model=plm,
                                            layer_nums = layer_nums, pooled=True, as_numpy=True)

        # Add the scrambled sequence embeddings to the ones we got from the cache
        for layer_num in embeddings.keys():
            embeddings[layer_num] = np.vstack([embeddings[layer_num], scrambled_embeddings[layer_num]])        

        # Do PCA on each layer and store results
        pca_layerwise = pca_layerwise_embeddings(embeddings)
        # Make an annotation df with the pcs we've calculated
        
        for layer_num in pca_layerwise.keys():
            pcs = pca_layerwise[layer_num]
            pc1 = pcs['pca_0']
            pc2 = pcs['pca_1']
            # Make a deep copy
            ann_df = annotation_df.clone()
            # Add PCA columns and layer num
            ann_df = ann_df.with_columns(
                pl.lit(layer_num).alias('layer_num'),
                pl.lit(dataset).alias('dataset'),
                pl.lit(model_shorthand).alias('model_name'),
                pl.Series(name='pc_1',
                values=pc1,
                dtype=pl.Float32),
                pl.Series(name='pc_2',
                values=pc2,
                dtype=pl.Float32),
            ).select(['id', 'sequence', 'combined_labels',	'label_num', 'layer_num', 'dataset', 'model_name', 'pc_1', 'pc_2'])
            layer_dfs.append(ann_df)
# Combine results for the different models
pca_df = pl.concat(layer_dfs)

In [ ]:
pca_df.write_parquet(pca_compiled_results_dir / 'compiled_pca_select_models_annotated_human_proteome.parquet.gz')

In [ ]:
pca_df

In [ ]:

# High-resolution output settings
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

col_by = 'layer_num'
row_by = 'model_name'
color_by = 'combined_labels'
# Using 3.5 MAD is roughly equivalent to 2.5 - 3 Standard Deviations
threshold_mad = 3.5 

for dataset in pca_df['dataset'].unique():
    unique_labels = sorted(pca_df.filter(pl.col('dataset') == dataset)[color_by].unique().to_list())
    distinct_colors = sns.color_palette(cc.glasbey, n_colors=len(unique_labels))
    custom_palette = {label: color for label, color in zip(unique_labels, distinct_colors)}
    custom_palette['scrambled'] = '#FF0000' 

    custom_sizes = {label: 25 for label in unique_labels}
    custom_sizes['scrambled'] = 75

    plot_data = pca_df.filter(pl.col('dataset') == dataset).to_pandas()
    plot_data['is_scrambled'] = plot_data[color_by] == 'scrambled'
    plot_data = plot_data.sort_values('is_scrambled') 

    fig = sns.relplot(
        data=plot_data, x='pc_1', y='pc_2',
        col=col_by, col_order=layer_nums,
        row=row_by, row_order=model_shorthands,
        kind='scatter', hue=color_by, size=color_by,       
        sizes=custom_sizes, alpha=0.7, linewidth=0.5, height=7,
        palette=custom_palette, facet_kws={'sharey': False, 'sharex': False},
        legend=False
    )

    for (row_val, col_val), ax in fig.axes_dict.items():
        facet_data = plot_data[(plot_data[row_by] == row_val) & (plot_data[col_by] == col_val)]
        if facet_data.empty: continue

        # Iterate through each group
        for label, group_df in facet_data.groupby(color_by):
            if len(group_df) < 5: continue 
            
            # --- LEAVE-ONE-OUT LOGIC ---
            # Create a 'background' dataframe that excludes the current group
            background = facet_data[facet_data[color_by] != label]
            
            if background.empty: # Fallback for single-group plots
                background = facet_data

            # Calculate robust centroid (Median) of the background
            bg_centroid = background[['pc_1', 'pc_2']].median().values
            
            # Calculate robust spread (MAD) of the background
            # MAD = median(|x - median(x)|)
            bg_median = background[['pc_1', 'pc_2']].median()
            bg_mad = (background[['pc_1', 'pc_2']] - bg_median).abs().median().values
            
            # Normalizing factor for MAD to make it comparable to StdDev (approx 1.4826)
            bg_mad = bg_mad * 1.4826 
            bg_mad = np.where(bg_mad == 0, 1e-6, bg_mad) # Prevent div by zero

            # Position of current group
            group_pos = group_df[['pc_1', 'pc_2']].mean().values
            
            # Calculate distance in "Robust Standard Deviations" (MAD units)
            # This measures how many background-spreads away the group is
            z_scores = (group_pos - bg_centroid) / bg_mad
            robust_dist = np.linalg.norm(z_scores)

            is_scrambled = 'scrambled' in str(label).lower()
            
            if robust_dist > threshold_mad or is_scrambled:
                label_color = custom_palette.get(label, 'black')
                display_name = label.split('__')[-1].split(';')[0].split('[')[0]
                
                ax.text(
                    group_pos[0], group_pos[1], display_name,
                    fontsize=13 if not is_scrambled else 11,
                    fontweight='bold', color=label_color,
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=0.1),
                    ha='center', va='center', zorder=30
                )

    fig.fig.suptitle(f"{dataset.replace('_', ' ').title()}", fontsize=18, fontweight='bold', y=1.02)
    fig.set_titles(col_template='Layer {col_name}', row_template='{row_name}')
    # Construct a descriptive filename
    save_filename = f"pca_facet_{dataset}_mad_{threshold_mad}_{SEED}.png"
    save_path = pca_figures_dir / save_filename

    # Save the figure
    fig.savefig(
        save_path, 
        dpi=300, 
        bbox_inches='tight', 
        facecolor='white', 
        transparent=False
    )
    plt.show()

We can see that there are clear groupings, most apparent for 'domain' and 'family' but for other categories as well. GPCRs and related proteins are often most easily recognized and grouped in the latent space of different models.